In [2]:
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import spearmanr
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from scipy import stats
from Lasso import select_features_lasso
from sklearn.inspection import permutation_importance

In [3]:
df = pd.read_csv("../datasets_prep/dm_project_dataset.csv")
df['observationTime'] = pd.to_datetime(df['observationTime'])
df.columns.to_list()

['observationTime',
 'absorbing_aerosol_index',
 'NO2_column_number_density',
 'stratospheric_NO2_column_number_density',
 'NO2_slant_column_number_density',
 'tropopause_pressure',
 'H2O_column_number_density',
 'BrO',
 'BrO_Error',
 'NO2',
 'NO2_Error',
 'O3',
 'O3_Error',
 'pm10',
 'pm2p5',
 'sia',
 'nmvoc',
 'dust',
 'nh3',
 'pans',
 'tp',
 't2m',
 'airTemperature',
 'seaLevelPressure',
 'relativeHumidity',
 'missforest_precipitation',
 'xgb_cloudCover',
 'feelsLikeTemperature_fixed',
 'windSpeed_fixed',
 'windGust_fixed',
 'windDirection_fixed']

In [4]:
cols_to_remove = ['BrO_Error', 'NO2_Error', 'O3_Error']
data = df.drop(columns = cols_to_remove)

In [5]:
data['month'] = data['observationTime'].dt.month
data['day_of_year'] = data['observationTime'].dt.dayofyear
data['day_of_week'] = data['observationTime'].dt.dayofweek

data['target_precipitation'] = data['missforest_precipitation'].shift(-1)
data = data.dropna(subset=['target_precipitation'])

In [6]:
train_end = '2024-02-01'
val_end = '2025-02-01'

target_col = 'target_precipitation'
drop_cols = ['missforest_precipitation','observationTime', 't2m', target_col]

In [7]:
def metrics_fun(y_val, rf_pred_val):
    print(f'R^2:  {r2_score(y_val, rf_pred_val)}')
    print(f'MAE:  {mean_absolute_error(y_val, rf_pred_val)}')
    print(f'RMSE: {np.sqrt(mean_squared_error(y_val, rf_pred_val))}')

    corr, p_value = stats.spearmanr(y_val, rf_pred_val)
    print(f"r_s:  {corr:.3f}")

In [8]:
def add_specific_lags(df):
    df = df.copy()
    
    df['airTemperature_lag10'] = df['airTemperature'].shift(10)
    df['relativeHumidity_lag_lag4'] = df['relativeHumidity'].shift(4)
    df['feelsLikeTemperature_fixed_lag3'] = df['feelsLikeTemperature_fixed'].shift(3)

    
    return df

data_with_lags = add_specific_lags(data)
data_with_lags = data_with_lags.dropna().reset_index(drop=True)

original_pollutants = [
    'airTemperature',
    'relativeHumidity',
    'feelsLikeTemperature_fixed',
    'absorbing_aerosol_index',
    'NO2_column_number_density',
    'stratospheric_NO2_column_number_density',
    'NO2_slant_column_number_density',
    'tropopause_pressure',
    'H2O_column_number_density',
    'BrO',
    'NO2',
    'O3',
    'pm10',
    'pm2p5',
    'sia',
    'nmvoc',
    'dust',
    'nh3',
    'pans',
    'airTemperature',
    # 'seaLevelPressure',
    'relativeHumidity',
    # 'xgb_cloudCover',
    'feelsLikeTemperature_fixed',
    # 'windSpeed_fixed',
    # 'windGust_fixed',
    # 'windDirection_fixed'
]

data_with_lags = data_with_lags.drop(columns=original_pollutants)

In [11]:
train = data_with_lags[data_with_lags['observationTime'] < train_end]
val   = data_with_lags[(data_with_lags['observationTime'] >= train_end) & (data_with_lags['observationTime'] <= val_end)]
test  = data_with_lags[data_with_lags['observationTime'] > val_end]

X_train = train.drop(columns=drop_cols)
y_train = train[target_col]

X_val = val.drop(columns=drop_cols)
y_val = val[target_col]

X_test = test.drop(columns=drop_cols)
y_test = test[target_col]

final_model = XGBRegressor(random_state=2026)
final_model.fit(X_train, y_train)

perm = permutation_importance(
    final_model, X_val, y_val,
    n_repeats=10,
    random_state=2026,
    scoring='neg_mean_absolute_error'
)

perm_df = pd.DataFrame({
    'feature':    X_val.columns,
    'importance': perm.importances_mean,
    'std':        perm.importances_std
}).sort_values('importance', ascending=False)

print("\nPermutation Importance:")
print(perm_df.to_string(index=False))

print("Val rezultatai:")
metrics_fun(y_val, final_model.predict(X_val))


Permutation Importance:
                        feature  importance      std
                             tp    0.912206 0.069766
feelsLikeTemperature_fixed_lag3    0.268853 0.045475
                 xgb_cloudCover    0.204549 0.051126
                    day_of_year    0.073818 0.040564
           airTemperature_lag10    0.070724 0.030178
      relativeHumidity_lag_lag4    0.069622 0.036645
                windSpeed_fixed    0.068211 0.033850
                 windGust_fixed    0.029953 0.032173
               seaLevelPressure    0.000349 0.061249
                    day_of_week   -0.005860 0.018056
                          month   -0.010238 0.005393
            windDirection_fixed   -0.016382 0.044275
Val rezultatai:
R^2:  0.18310897133102677
MAE:  1.9075227252910947
RMSE: 4.364172393599264
r_s:  0.585


In [25]:
tscv = TimeSeriesSplit(n_splits=5)

# param_grid = {
#     'n_estimators': [700, 750, 800, 850, 900],
#     'learning_rate': [0.001, 0.003, 0.005, 0.006, 0.01],
#     'max_depth': [2, 3, 4, 6],
#     'subsample': [0.5, 0.55, 0.6, 0.65, 0.7],
#     'min_split_loss':[14, 16, 18, 20]
# }

# param_grid = {
#     'n_estimators': [700, 800, 900, 1000, 1100, 1200],
#     'learning_rate': [0.003, 0.005, 0.006],
#     'max_depth': [4, 5, 6, 7, 8],
#     'subsample': [0.2, 0.3, 0.4],
#     'min_split_loss':[26, 28, 30, 32, 34, 36, 38]
# }

param_grid = {
    'n_estimators': [300, 350, 400, 450, 500],
    'learning_rate': [0.001, 0.005, 0.01, 0.05, 0.1],
    'max_depth': [2, 3, 5],
    'subsample': [0.3, 0.4, 0.5, 0.6, 0.7]
}

xgb = XGBRegressor(random_state=2026)

grid_search = GridSearchCV(
    estimator=xgb, 
    param_grid=param_grid, 
    cv=tscv, 
    scoring='neg_root_mean_squared_error',
    verbose=1,
    n_jobs=-1 
)

grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_

print(f"Geriausi parametrai: {grid_search.best_params_}")

Fitting 5 folds for each of 375 candidates, totalling 1875 fits
Geriausi parametrai: {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 400, 'subsample': 0.5}


In [26]:
val_preds = model.predict(X_val)

mae = mean_absolute_error(y_val, val_preds)
rmse = np.sqrt(mean_squared_error(y_val, val_preds))
r2 = r2_score(y_val, val_preds)
corr, p_value = spearmanr(y_val, val_preds)

print(f"MAE: {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2: {r2:.4f}")
print(f"Spearmano koreliacijos koeficientas: {corr:.3f}")

MAE: 1.6036
RMSE: 3.7494
R2: 0.3971
Spearmano koreliacijos koeficientas: 0.697


In [27]:
train_preds = model.predict(X_train).clip(min=0)
val_preds   = model.predict(X_val).clip(min=0)
test_preds  = model.predict(X_test).clip(min=0)

train_results = pd.DataFrame({
    'observationTime': train['observationTime'] + pd.Timedelta(days=1),
    'actual': y_train,
    'xgb_predicted': model.predict(X_train),
})

val_results = pd.DataFrame({
    'observationTime': val['observationTime'] + pd.Timedelta(days=1),
    'actual': y_val,
    'xgb_predicted': val_preds,
})

test_results = pd.DataFrame({
    'observationTime': test['observationTime'] + pd.Timedelta(days=1),
    'actual': y_test,
    'xgb_predicted': test_preds,
})

all_results = pd.concat([train_results, val_results, test_results], ignore_index=True)
all_results.to_csv('xgboost_pred_no_pollution.csv', index=False)

print(all_results.head())

  observationTime  actual  xgb_predicted
0      2018-07-12     0.3       1.405656
1      2018-07-13    24.0      17.141331
2      2018-07-14    13.3      18.700474
3      2018-07-15    11.1       7.930111
4      2018-07-16     4.8       2.179455
